<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/11_2_Role_based_Multi_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습 11-2] 역할 기반 멀티 에이전트 아키텍처 구현  

### 실습목표

- 페르소나(Persona) 설계를 통해 역할과 도구가 분리된 전문 에이전트들을 생성할 수 있다.  

- 순차적(Sequential) 및 계층적(Hierarchical) 토폴로지를 코드로 구현하여 데이터 흐름의 차이를 이해한다.  

- Manager(Supervisor) 에이전트를 활용한 중앙 조율(Orchestration) 구조를 설계하고 자율적인 업무 할당 과정을 검증한다.  

1. 환경 준비 및 라이브러리 설치  

- 실습을 위해 LangChain 및 Google Gemini 설정을 진행합니다.  

In [2]:
# 실습을 위한 라이브러리 설치
!pip install -q -U langchain langchain-google-genai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 13.3 MB/s eta 0:00:00


In [8]:
!pip install -q -U google-generativeai

# Google API Key 설정
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

Gemini API 설정 완료


In [5]:
# 1. 필수 라이브러리 설치
# !pip install -q -U langchain langchain-google-genai langchain-core

import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.tools import tool
import textwrap

# Gemini API 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

2. [파트 1] 역할별 전문 에이전트 및 도구 분리 설계  

- 모든 에이전트에게 모든 도구를 주지 않고, 역할에 맞춰 도구(Tool)를 분리 할당하여 추론의 정확도를 높입니다.  

In [6]:
# --- [도구 정의: Researcher 전용] ---
@tool
def search_latest_tech(query: str) -> str:
    """최신 IT 기술 동향 및 실시간 정보를 검색합니다."""
    # 시뮬레이션 데이터 제공
    tech_data = {
        "멀티 에이전트": "2026년 표준: 에이전트 간 '자율 협업 규약'이 확립되어 이기종 간 통신이 가능해짐",
        "오케스트레이션": "중앙 집중형 매니저 구조에서 분산형 자율 협업 구조로 진화 중"
    }
    for key in tech_data:
        if key in query: return f"[검색 결과] {tech_data[key]}"
    return "[검색 결과] 관련 최신 기술이 활발히 연구되고 있습니다."

# --- [에이전트 페르소나 설계] ---

# 1. 기획 에이전트 (Planner): 도구 없이 '전략 수립'에만 집중
planner_prompt = ChatPromptTemplate.from_template("""
역할: 콘텐츠 전략 기획자
목표: {topic}에 대한 창의적이고 논리적인 목차를 구성하세요.
지침: 본문은 작성하지 말고 목차와 핵심 키워드만 설계하세요.
""")
planner_agent = planner_prompt | llm | StrOutputParser()

# 2. 분석 에이전트 (Researcher): '검색 도구'를 활용하여 팩트 수집 및 본문 작성
researcher_llm = llm.bind_tools([search_latest_tech])
researcher_prompt = ChatPromptTemplate.from_template("""
역할: 기술 분석 전문가
목표: 전달받은 [기획안]을 바탕으로 검색 도구를 사용하여 구체적인 정보를 수집 및 요약하여 본문을 완성하세요.
기획안: {plan}
""")

3. [파트 2] 중앙 조율(Orchestration) 로직 구현  

- Manager 역할을 하는 함수가 에이전트들의 실행 순서와 종료 조건을 관리하며, 가독성 있게 출력합니다.  

In [9]:
def run_integrated_orchestrator(user_request):
    print(f"\n[Manager] 워크플로우 조율 시작: {user_request}")
    print("=" * 60)

    # 1. Task Delegation: 기획자에게 전략 수립 요청
    print("[Manager] 단계 1: Planner에게 기획안 작성을 지시합니다.")
    plan = planner_agent.invoke({"topic": user_request})

    # 2. Quality Check & Communication: 기획안 확인 후 분석가에게 전달
    if len(plan) > 30:
        print(f"[*] 기획안 승인 완료 (Memory 저장). Researcher에게 분석을 지시합니다.")

        # Researcher 실행 및 도구 결과 처리
        # researcher_prompt를 사용하여 메시지를 구성하고 researcher_llm을 호출
        researcher_messages = researcher_prompt.format_messages(plan=plan)
        response = researcher_llm.invoke(researcher_messages)

        # response.content가 리스트일 경우 텍스트만 추출하여 문자열로 변환
        if isinstance(response.content, list):
            final_report = " ".join([part['text'] for part in response.content if part.get('type') == 'text'])
        else:
            final_report = response.content
    else:
        print("[Manager] 기획안 품질 미달로 프로세스를 중단합니다.")
        return

    # 3. Readability Optimization: 가독성 최적화 출력
    print("\n[최종 요약 보고서]")
    print("-" * 60)

    # 70자 기준으로 줄바꿈 처리하여 화면에 맞춤
    wrapped_report = textwrap.fill(final_report, width=70, break_long_words=False, replace_whitespace=False)
    print(wrapped_report)
    print("-" * 60)
    print("[Manager] 모든 작업이 성공적으로 완료되었습니다. (Stop Condition 충족)")

# 실행 테스트
run_integrated_orchestrator("미래 AI 에이전트의 오케스트레이션 전망")


[Manager] 워크플로우 조율 시작: 미래 AI 에이전트의 오케스트레이션 전망
[Manager] 단계 1: Planner에게 기획안 작성을 지시합니다.
[*] 기획안 승인 완료 (Memory 저장). Researcher에게 분석을 지시합니다.

[최종 요약 보고서]
------------------------------------------------------------

------------------------------------------------------------
[Manager] 모든 작업이 성공적으로 완료되었습니다. (Stop Condition 충족)


- **실습 결과 비교 및 분석**  

    - 역할 분리 효과: "Planner는 목차 설계에, Researcher는 팩트 수집에만 집중하여 전문성 향상"  
    - 도구 할당 효과: 특정 에이전트에게만 도구를 주어 모델이 불필요한 도구 호출을 고민하지 않음  
    - 조율(Orchestration): Manager 역할을 통해 작업의 순서와 종료 시점(Stop Condition)을 유연하게 통제  

### 학습점검

- Q1. 왜 모든 에이전트에게 모든 도구(Tool)를 주지 않고 분리하여 할당하나요?  
    - A. 모델이 처리해야 할 프롬프트 길이를 줄여 추론 오류를 방지하고, 각 역할에만 집중하게 하여 전문성을 높이기 위함입니다.  

- Q2. 계층적 구조(Manager)가 순차적 구조보다 유리한 상황은 언제인가요?  
    - A. 작업 순서가 고정되지 않고, 이전 작업의 결과물 상태에 따라 다음에 수행할 작업을 동적으로 결정해야 하는 복잡한 상황에 유리합니다.  